# 20260721_EDA_2016_전국_시도분리정제

- 작성자: 이정연
- #17 이슈 참고 

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    clean_sentence,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    SUBTOTAL_LABEL_PATTERN,
    UNIT_NOTATION_PATTERN,
    assign_labels,
    backfill_major_category_from_medium,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [2]:
YEAR = 2016
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2016년도 지방자치단체 저출산고령사회 시행계획 (제3차 기본계획)_칼럼정렬.xlsx"
)
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = SOURCE_FILE  # Table 1 시트가 같은 파일 안에 있음

ZERO_BUDGET_TOKENS = ("-",)
# 2016은 시도별 "(단위 : 백만원)" 헤더 행(이슈 #24)과 총계/소계/합계 라벨 행(이슈 #29 예정)이 있음
EXTRA_HEADER_PATTERNS = (UNIT_NOTATION_PATTERN, SUBTOTAL_LABEL_PATTERN)
GWANGJU_AGGREGATE_LABELS = {"공통사업+자체사업", "공통사업 (저출산+고령사회)"}
DAEJEON_MAJOR_LABEL_PATTERN = r"^\s*\[\s*(?:공\s*통|자\s*체)\s*사\s*업\s*\]"
DAEJEON_MEDIUM_LABEL_PATTERN = r"^[Ⅰ-Ⅿ]"
DAEJEON_LOWER_SUBTOTAL_PATTERN = r"\(\s*\d+\s*개\s*과제\s*\)\s*$"
# 2016은 사업분류재정구분이 계/국비/지방비 등 재원 구분이라 이중계상 위험이 있음 (이슈 #26)
NEEDS_FUNDING_SOURCE_CLEANUP = True
# 원본 표에서 추출 오류임을 직접 확인한 삭제 대상만 지정한다. 현재 확인된 대상 없음.
CONFIRMED_DUPLICATE_ROWS = ()
QA_TOLERANCE = 0
RUN_LLM = True

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"

## 2. 데이터 로드와 기본 확인

In [3]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

print("데이터 크기:", df_raw.shape)
print("지역 수:", df_raw["지역"].nunique())
display(df_raw.head())

데이터 크기: (6882, 12)
지역 수: 17


,지역,세부사업명,사업분류재정구분,2016년 예산,2015년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,(단위 : 백만원),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,서울,총 계,NaN,4173069,4480127,-307058,-6.9,NaN,1.0,2.0,4.0,데이터행
2,서울,Ⅰ. 공통사업,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,5.0,구분행사업명 연속 후보
3,서울,공통사업 합계,계,3772641,4075595,-302954,-7.4,NaN,1.0,2.0,6.0,데이터행
4,서울,공통사업 합계,국비,1775591,2282149,-506558,-22.2,NaN,1.0,2.0,7.0,데이터행


## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [4]:
if NEEDS_FUNDING_SOURCE_CLEANUP:
    df_raw = drop_exact_duplicate_rows(df_raw, confirmed_duplicate_rows=CONFIRMED_DUPLICATE_ROWS)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# 광주의 두 행은 공통·자체사업을 합산한 총괄 행이므로 세부사업 합계에서 제외한다.
is_gwangju_aggregate = df_raw["지역"].eq("광주") & df_raw["세부사업명"].isin(
    GWANGJU_AGGREGATE_LABELS
)
df_raw.loc[is_gwangju_aggregate, "사업행구분"] = "헤더반복"

# 경기 원본행 3162~3175는 전체 예산 요약 블록이다. 대분류 전파 기준인 3165는 유지한다.
is_gyeonggi_summary = (
    df_raw["지역"].eq("경기") & df_raw["원본행"].between(3162, 3175) & ~df_raw["원본행"].eq(3165)
)
df_raw.loc[is_gyeonggi_summary, "사업행구분"] = "헤더반복"

# 대구 원본행 1095·1096은 공통사업 합계의 국비·지방비 상세 행이다.
is_daegu_common_summary_detail = df_raw["지역"].eq("대구") & df_raw["원본행"].isin([1095, 1096])
df_raw.loc[is_daegu_common_summary_detail, "사업행구분"] = "헤더반복"

# 대전은 대괄호 제목이 대분류, 로마숫자 제목이 중분류이고 (N개 과제) 제목이 하위 소계다.
is_daejeon = df_raw["지역"].eq("대전")
daejeon_detail_name = df_raw["세부사업명"].astype("string").str.strip()
is_daejeon_major = is_daejeon & daejeon_detail_name.str.contains(
    DAEJEON_MAJOR_LABEL_PATTERN, regex=True, na=False
)
is_daejeon_medium = is_daejeon & daejeon_detail_name.str.contains(
    DAEJEON_MEDIUM_LABEL_PATTERN, regex=True, na=False
)
is_daejeon_lower_subtotal = (
    is_daejeon
    & ~is_daejeon_major
    & ~is_daejeon_medium
    & daejeon_detail_name.str.contains(DAEJEON_LOWER_SUBTOTAL_PATTERN, regex=True, na=False)
)
df_raw.loc[is_daejeon_major, "사업행구분"] = "대분류_소계"
df_raw.loc[is_daejeon_medium, "사업행구분"] = "중분류_소계"
df_raw.loc[is_daejeon_lower_subtotal, "사업행구분"] = "헤더반복"

# TODO: 특정 연도에 예산이 모두 결측인 페이지 머리글이 있다면 여기서 헤더반복으로 보정

df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]
df_labeled = backfill_major_category_from_medium(df_labeled)
is_daejeon_labeled = df_labeled["지역"].eq("대전")
df_labeled.loc[
    is_daejeon_labeled & df_labeled["대분류"].str.contains(r"공\s*통", na=False),
    "대분류",
] = "공통사업"
df_labeled.loc[
    is_daejeon_labeled & df_labeled["대분류"].str.contains(r"자\s*체", na=False),
    "대분류",
] = "자체사업"

# 광주·대전의 대분류 표기를 다른 지역과 동일한 최종 라벨로 통일한다.
MAJOR_CATEGORY_NORMALIZATION = {
    "공통사업": "Ⅰ. 공통사업",
    "자체사업": "Ⅱ. 지자체 자체사업",
}
df_labeled["대분류"] = df_labeled["대분류"].replace(MAJOR_CATEGORY_NORMALIZATION)
df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

display(df_labeled.head())

재원구분(계/국비/지방비) 정리 후 행 수: 4802
사업행구분
세부사업      4494
헤더반복       181
중분류_소계      95
대분류_소계      32
Name: count, dtype: int64
세부사업 행 수: 4494


,세부사업명,사업분류재정구분,2016년 예산,2015년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,재원구분,사업행구분,대분류,중분류,지역,2016년_예산_num,2015년_예산_num
141,(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),144.0,3446.0,3463.0,데이터행,<NA>,헤더반복,NaN,NaN,강원,<NA>,<NA>
142,총 계,NaN,966599,952834,13765,1.4,NaN,145.0,3464.0,3466.0,데이터행,<NA>,헤더반복,NaN,NaN,강원,966599.0,952834.0
143,Ⅰ. 공통사업,NaN,NaN,NaN,NaN,NaN,NaN,145.0,3464.0,3467.0,구분행사업명 연속 후보,<NA>,대분류_소계,Ⅰ. 공통사업,NaN,강원,<NA>,<NA>
752,공통사업 합계,계,836613,830867,5746,0.7,NaN,145.0,3464.0,3468.0,데이터행,계,헤더반복,Ⅰ. 공통사업,NaN,강원,836613.0,830867.0
144,1. 저출산 분야(공통사업),NaN,NaN,NaN,NaN,NaN,NaN,145.0,3464.0,3471.0,구분행사업명 연속 후보,<NA>,중분류_소계,Ⅰ. 공통사업,1. 저출산 분야(공통사업),강원,<NA>,<NA>


## 4. 텍스트 정제와 예산 재계산

In [5]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head(20))

,지역,세부사업명,당해예산,전년도예산,증감액,증감율
754,강원,보육비 지원확대,117814.0,123860.0,-6046.0,-4.9
755,강원,만3~5세 누리과정 확대,98756.0,109808.0,-11052.0,-10.1
756,강원,양육수당 지원 확대,43412.0,43186.0,226.0,0.5
757,강원,보육교지원 인건비 지원,60922.0,52648.0,8274.0,15.7
758,강원,공공형어린이집 지원,4600.0,4436.0,164.0,3.7
759,강원,어린이집 환경개선,930.0,928.0,2.0,0.2
760,강원,육아종합지원센터 운영 지원,548.0,380.0,168.0,44.2
761,강원,방과후과정 교육비 지원,12259.0,15471.0,-3212.0,-20.8
762,강원,입양가정 양육수당 지원,911.0,1131.0,-220.0,-19.5
763,강원,지역아동센터 확대 및 내실화,12618.0,12016.0,602.0,5.0


In [6]:
# 지역마다 유니크 대분류가 몇개고 종류가 뭔지 확인
major_by_region = df_leaf.groupby("지역")["대분류"].agg(
    대분류_소계="nunique", 대분류_종류=lambda s: sorted(s.dropna().unique())
)

major_by_region

,대분류_소계,대분류_종류
지역,,
강원,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
경기,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
경남,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
경북,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
광주,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
대구,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
대전,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
부산,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"
서울,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]"


In [7]:
# 유틸 함수로 대분류가 전파되지 않은 경우
unsolvedd_major = df_leaf[df_leaf["대분류"].isna()]
print("대분류 미전파 건수: ", len(unsolvedd_major))
print(unsolvedd_major["지역"].value_counts())

display(unsolvedd_major[["지역", "세부사업명", "원본행"]])

대분류 미전파 건수:  0
Series([], Name: count, dtype: int64)


,지역,세부사업명,원본행


### 4-1. 사업분류재정구분에 공통/자체 파싱
- 대분류 텍스트에서 공통/자체를 파싱한다. (4차 시행계획 정제 스키마와 통일하는 작업)

In [8]:
is_common = df_leaf["대분류"].str.contains("공통", na=False)
is_self = df_leaf["대분류"].str.contains("자체", na=False)

df_leaf["사업분류재정구분"] = pd.NA  # 초기화
df_leaf.loc[is_common, "사업분류재정구분"] = "공통"
df_leaf.loc[is_self, "사업분류재정구분"] = "자체"

print(df_leaf["사업분류재정구분"].value_counts(dropna=False))
display(df_leaf[df_leaf["사업분류재정구분"].isna()].head())

사업분류재정구분
자체    3535
공통     959
Name: count, dtype: int64


,세부사업명,사업분류재정구분,2016년 예산,2015년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,재원구분,사업행구분,대분류,중분류,지역,2016년_예산_num,2015년_예산_num,당해예산,전년도예산,증감율


## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [9]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
    subtotal_label_col="세부사업명",
    subtotal_labels=("소계",),
)

display(qa["결과"].value_counts(dropna=False))
display(qa[qa["결과"] == "불일치"])

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

결과
일치      51
불일치     42
판정불가     2
Name: count, dtype: int64

,지역,대분류,중분류,원본_소계값,원본_소계출처,leaf_합계,차이,절대차이,오차율(%),절대오차율(%),QA_병합상태,결과,허용기준결과
2,강원,Ⅰ. 공통사업,3. 대응기반 강화분야(공통사업),8914.0,별도소계행,8643.0,-271.0,271.0,-3.04,3.04,양쪽존재,불일치,허용
3,강원,Ⅱ. 지자체 자체사업,1. 저출산분야(자체사업),47666.0,별도소계행,47667.0,1.0,1.0,0.0,0.0,양쪽존재,불일치,허용
4,강원,Ⅱ. 지자체 자체사업,2. 고령사회분야(자체사업),80764.0,별도소계행,80765.0,1.0,1.0,0.0,0.0,양쪽존재,불일치,허용
12,경남,Ⅰ. 공통사업,1. 저출산 대책(공통사업),391442.0,별도소계행,391567.0,125.0,125.0,0.03,0.03,양쪽존재,불일치,허용
15,경남,Ⅱ. 지자체 자체사업,1. 저출산 대책(자체사업),68846.0,별도소계행,68847.0,1.0,1.0,0.0,0.0,양쪽존재,불일치,허용
16,경남,Ⅱ. 지자체 자체사업,2. 고령화 대책(자체사업),19820.0,별도소계행,19821.0,1.0,1.0,0.01,0.01,양쪽존재,불일치,허용
20,경북,Ⅱ. 지자체 자체사업,1. 저출산 대책(자체사업),392613.0,별도소계행,392618.0,5.0,5.0,0.0,0.0,양쪽존재,불일치,허용
21,경북,Ⅱ. 지자체 자체사업,2. 고령화 대책(자체사업),51310.0,별도소계행,51311.0,1.0,1.0,0.0,0.0,양쪽존재,불일치,허용
22,경북,Ⅱ. 지자체 자체사업,3. 대응기반 강화(자체사업),896.0,별도소계행,897.0,1.0,1.0,0.11,0.11,양쪽존재,불일치,허용
25,광주,Ⅰ. 공통사업,저출산 대책,324012.0,중분류제목행,648349.0,324337.0,324337.0,100.1,100.1,양쪽존재,불일치,초과


QA 저장 완료: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/reports/2016_전국_QA_검증결과.csv


## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [10]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

유사 사업명 검토 후보: 121


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
0,전남,1. 저출산 대책(공통사업),저소득층 기저귀조제분유 지원사업,20.0,"저소득층 영유아 가정 기저귀,조제분유 지원",저소득층 기저귀･조제분유 지원사업,26.0,저소득층 기저귀 및 분유 지원사업 - 중위소득 80% 이하인 자 - 출산 예정일 40일전부터 출산 후 30일 까지,97.142857
1,전남,1. 저출산 대책(공통사업),산모신생아 건강관리 지원,54.0,출산가정에 도우미서비스지원을 통한 산모 및 신생아 건강관리 도모(41명),산모신생아 건강관리사 지원,89.0,출산가정에 도우미 서비스지원을 통한 산모 및 신생아 건강관리 도모,96.296296
2,전남,1. 저출산 대책(공통사업),산모신생아 건강관리 지원,54.0,출산가정에 도우미서비스지원을 통한 산모 및 신생아 건강관리 도모(41명),산모신생아 건강관리사 지원,118.0,산모 200여명에게 산모신생아 2주간 돌봄서비스,96.296296
3,전남,1. 저출산 대책(공통사업),산모신생아 건강관리 지원,2442.0,"출산가정에 도우미서비스 지원을 통한 산모 및 신생아 건강관리 도모(4,140명)",산모신생아 건강관리사 지원,89.0,출산가정에 도우미 서비스지원을 통한 산모 및 신생아 건강관리 도모,96.296296
4,전남,1. 저출산 대책(공통사업),산모신생아 건강관리 지원,2442.0,"출산가정에 도우미서비스 지원을 통한 산모 및 신생아 건강관리 도모(4,140명)",산모신생아 건강관리사 지원,118.0,산모 200여명에게 산모신생아 2주간 돌봄서비스,96.296296
5,충북,1. 저출산 대책(자체사업),어린이집 아동간식 지원,175.0,학교 급식지원사업과 연계하여 어린이집 아동간식 지원,어린이집 아동간식비 지원,110.0,어린이집 아동간식비 지원,96.000000
6,충남,1. 저출산 대책(자체사업),농어촌 방과후학교 지원,2709.0,"농어촌지역 방과후 학교 프로그램 운영비, 강사비, 재료비 등 지원",농산어촌 방과후학교 지원,200.0,농산어촌지역 44개교 지원,96.000000
7,대전,"Ⅱ. 안정된 생활, 건강한 노년, 활기찬 노후 (50개 과제)",경로식당 급식도우미 운영,29.0,"25개소, 78명",경로당 급식도우미 운영,6284.0,"1일 2시간, 월20일 근무 • 식사준비, 청소등 위생관리",96.000000
8,부산,1. 저출산 대책(자체사업),어린이집 냉난방비 지원,15.5,직장어린이집을 제외한 어린이집 보육 현원별 냉난방비 차등 지원,어린이집 냉방비 지원,11.0,어린이집 냉방비 지원,95.652174
9,부산,1. 저출산 대책(자체사업),어린이집 냉난방비 지원,15.5,직장어린이집을 제외한 어린이집 보육 현원별 냉난방비 차등 지원,어린이집 난방비 지원,76.0,어린이집에 동절기 난방비 지원,95.652174


## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [11]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정(선택)

`RUN_LLM=True`로 바꾸기 전에 `.env`, 설정 파일, 연도별 체크포인트 파일명을 확인한다. 아래 셀은 연결 예시이며, 대량 실행 시 기존 노트북의 청크 단위 체크포인트 저장 로직을 함께 사용한다.

In [12]:
if RUN_LLM:
    import os
    from concurrent.futures import ThreadPoolExecutor
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(api_key=api_key, base_url=llm_cfg["upstage"]["base_url"])
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)

    chunk_size = 100
    if CHECKPOINT_PATH.exists():
        checkpoint_df = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig", index_col=0)
        checkpoint_df.index = pd.to_numeric(checkpoint_df.index, errors="raise").astype(int)
        checkpoint_df = checkpoint_df.loc[checkpoint_df.index.intersection(df_leaf.index)].copy()
        rerun_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
        checkpoint_df = checkpoint_df.loc[~rerun_mask].copy()
        print(f"체크포인트 발견: {len(checkpoint_df)}건 유지, {int(rerun_mask.sum())}건 재실행")
    else:
        checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
        print("체크포인트 없음: 전체 신규 실행")

    done_index = set(checkpoint_df.index)
    targets = [idx for idx in df_leaf.index if idx not in done_index]
    print(f"LLM 처리 대상: {len(targets)}건 / 전체 leaf {len(df_leaf)}건")
    results = {}

    def clean_target(idx):
        row = df_leaf.loc[idx]
        cleaned = clean_sentence(row["세부사업명"], row["주요내용"], call_once=call_once)
        return idx, cleaned

    with ThreadPoolExecutor(max_workers=6) as executor:
        cleaned_targets = executor.map(clean_target, targets)
        for processed, (idx, cleaned) in enumerate(cleaned_targets, start=1):
            results[idx] = cleaned

            if processed % chunk_size == 0:
                partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
                checkpoint_df = pd.concat([checkpoint_df, partial])
                checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
                results = {}
                print(f"체크포인트 저장: {processed}/{len(targets)}건")

    if results:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    checkpoint_df = checkpoint_df.loc[~checkpoint_df.index.duplicated(keep="last")]
    checkpoint_df = checkpoint_df.reindex(df_leaf.index)
    df_leaf["주요내용_정제"] = checkpoint_df["주요내용_정제"]
    df_leaf["숫자보존"] = [
        numbers_preserved(original, cleaned)
        for original, cleaned in zip(df_leaf["주요내용"], df_leaf["주요내용_정제"])
    ]

    bad_index = df_leaf.index[~df_leaf["숫자보존"]]
    if len(bad_index) > 0:
        review_cols = ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"]
        df_leaf.loc[bad_index, review_cols].to_csv(
            REPORTS_DIR / f"{YEAR}_LLM_숫자불일치_검토.csv",
            index=False,
            encoding="utf-8-sig",
        )
        df_leaf.loc[bad_index, "주요내용_정제"] = df_leaf.loc[bad_index, "주요내용"]
        checkpoint_df.loc[bad_index, "주요내용_정제"] = df_leaf.loc[bad_index, "주요내용"]
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    print(f"숫자 불일치 원문 복구: {len(bad_index)}건")
else:
    print("RUN_LLM=False: LLM 호출을 건너뜁니다.")

체크포인트 발견: 4494건 유지, 0건 재실행
LLM 처리 대상: 0건 / 전체 leaf 4494건
숫자 불일치 원문 복구: 0건


## 9. 시도별 최종 저장

최종 컬럼은 해당 연도 작업에서 확정한 스키마로 수정한다. 같은 파일을 중간과 최종 단계에서 중복 저장하지 않는다.

In [13]:
df_leaf["연도"] = YEAR

output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "재원구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
missing_output_cols = set(output_cols).difference(df_leaf.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

for sido, group in df_leaf.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    raw_output_path = sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    df_labeled.loc[df_labeled["지역"].eq(sido)].to_csv(
        raw_output_path, index=False, encoding="utf-8-sig"
    )
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

강원: 251행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/강원/2016_강원_세부사업_정제.csv
경기: 126행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경기/2016_경기_세부사업_정제.csv
경남: 319행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경남/2016_경남_세부사업_정제.csv
경북: 417행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경북/2016_경북_세부사업_정제.csv
광주: 278행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/광주/2016_광주_세부사업_정제.csv
대구: 160행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대구/2016_대구_세부사업_정제.csv
대전: 230행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대전/2016_대전_세부사업_정제.csv
부산: 552행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/부산/2016_부산_세부사업_정제.csv
서울: 103행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정

## 9. Long 포맷 변환 및 저장

당해예산과 전년도예산을 연도별 행으로 변환하고 시도별 파일로 저장한다.

In [14]:
long_id_cols = [column for column in output_cols if column not in {"당해예산", "전년도예산"}]
df_long = df_leaf[output_cols].melt(
    id_vars=long_id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

previous_mask = df_long["예산구분"].eq("전년도예산")
df_long.loc[previous_mask, "연도"] -= 1
df_long.loc[previous_mask, ["증감액", "증감율"]] = None
df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

assert len(df_long) == len(df_leaf) * 2
print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf), "행 x 2)")
display(df_long.head(10))

long 변환 결과: 8988 행 (wide 4494 행 x 2)


,연도,지역,대분류,중분류,사업분류재정구분,재원구분,세부사업명,주요내용,주요내용_정제,증감액,증감율,원본행,지원대상,지원내용_상세,예산구분,예산액
0,2016,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,보육비 지원확대,"영유아보육료(0~2세) 전계층지원 - 사업량 : 22천명 -0세 406천원, 1세 357천원, 2세 295천원","영유아보육료(0~2세) 전계층지원 - 사업량 : 22천명 -0세 406천원, 1세 357천원, 2세 295천원",-6046.0,-4.9,3475.0,NaN,NaN,당해예산,117814.0
1,2015,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,보육비 지원확대,"영유아보육료(0~2세) 전계층지원 - 사업량 : 22천명 -0세 406천원, 1세 357천원, 2세 295천원","영유아보육료(0~2세) 전계층지원 - 사업량 : 22천명 -0세 406천원, 1세 357천원, 2세 295천원",<NA>,<NA>,3475.0,NaN,NaN,전년도예산,123860.0
2,2016,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,만3~5세 누리과정 확대,"누리과정(3~5세) 전계층 지원 - 사업량 : 21천명 -1인당 월 290천원(보육료 220, 운영비 70)","누리과정(3~5세) 전계층 지원 - 사업량 : 21천명 -1인당 월 290천원(보육료 220, 운영비 70)",-11052.0,-10.1,3478.0,NaN,NaN,당해예산,98756.0
3,2015,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,만3~5세 누리과정 확대,"누리과정(3~5세) 전계층 지원 - 사업량 : 21천명 -1인당 월 290천원(보육료 220, 운영비 70)","누리과정(3~5세) 전계층 지원 - 사업량 : 21천명 -1인당 월 290천원(보육료 220, 운영비 70)",<NA>,<NA>,3478.0,NaN,NaN,전년도예산,109808.0
4,2016,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,양육수당 지원 확대,"어린이집 미이용아동 전계층 양육수당 지원 - 사업량 : 28천명 -0세 200천원, 1세 150천원, 2세 100천원","어린이집 미이용아동 전계층 양육수당 지원 - 사업량 : 28천명 -0세 200천원, 1세 150천원, 2세 100천원",226.0,0.5,3481.0,NaN,NaN,당해예산,43412.0
5,2015,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,양육수당 지원 확대,"어린이집 미이용아동 전계층 양육수당 지원 - 사업량 : 28천명 -0세 200천원, 1세 150천원, 2세 100천원","어린이집 미이용아동 전계층 양육수당 지원 - 사업량 : 28천명 -0세 200천원, 1세 150천원, 2세 100천원",<NA>,<NA>,3481.0,NaN,NaN,전년도예산,43186.0
6,2016,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,보육교지원 인건비 지원,"어린이집 인건비 지원 - 교직원, 대체교사, 보조교사 인건비 -민간어린이집 교재교구비, 환경개선비, 농어촌차량 운영비 등","어린이집 인건비 지원 - 교직원, 대체교사, 보조교사 인건비 - 민간어린이집 교재교구비, 환경개선비, 농어촌차량 운영비 등",8274.0,15.7,3484.0,NaN,NaN,당해예산,60922.0
7,2015,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,보육교지원 인건비 지원,"어린이집 인건비 지원 - 교직원, 대체교사, 보조교사 인건비 -민간어린이집 교재교구비, 환경개선비, 농어촌차량 운영비 등","어린이집 인건비 지원 - 교직원, 대체교사, 보조교사 인건비 - 민간어린이집 교재교구비, 환경개선비, 농어촌차량 운영비 등",<NA>,<NA>,3484.0,NaN,NaN,전년도예산,52648.0
8,2016,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,공공형어린이집 지원,공공형 어린이집 확충 : 98개소→108개소 • 공공형 어린이집 지원 : 규모별 차등 지원,공공형 어린이집 확충 : 98개소→108개소 • 공공형 어린이집 지원 : 규모별 차등 지원,164.0,3.7,3487.0,NaN,NaN,당해예산,4600.0
9,2015,강원,Ⅰ. 공통사업,1. 저출산 분야(공통사업),공통,계,공공형어린이집 지원,공공형 어린이집 확충 : 98개소→108개소 • 공공형 어린이집 지원 : 규모별 차등 지원,공공형 어린이집 확충 : 98개소→108개소 • 공공형 어린이집 지원 : 규모별 차등 지원,<NA>,<NA>,3487.0,NaN,NaN,전년도예산,4436.0


In [15]:
for sido, group in df_long.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제_long.csv"
    group.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

강원: 502행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/강원/2016_강원_세부사업_정제_long.csv
경기: 252행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경기/2016_경기_세부사업_정제_long.csv
경남: 638행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경남/2016_경남_세부사업_정제_long.csv
경북: 834행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경북/2016_경북_세부사업_정제_long.csv
광주: 556행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/광주/2016_광주_세부사업_정제_long.csv
대구: 320행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대구/2016_대구_세부사업_정제_long.csv
대전: 460행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대전/2016_대전_세부사업_정제_long.csv
부산: 1104행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/부산/2016_부산_세부사업_정제_long.csv
서울: 206행 저장 -> /Users/l

## 완료 전 체크리스트

- [ ] 17개 시도 포함 여부 확인
- [ ] 행 유형 분포 확인
- [ ] 연도 고유 헤더·연속행 후보 원본 대조
- [ ] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영 (2016~2019)
- [ ] 중분류 소계 QA 불일치 원인 기록
- [ ] 증감액·증감율 재계산 결과 확인
- [ ] LLM 숫자 보존 불일치 원본 대조
- [ ] wide/long 행 수와 컬럼 스키마 확인
- [ ] 최종 CSV `utf-8-sig` 저장 확인
- [ ] 커널 재시작 후 전체 실행
- [ ] 리팩터링 전후 QA 수치 비교